# Hidden Markov Model for Human Activity Recognition

This notebook implements a Hidden Markov Model to classify human activities (Standing, Walking, Jumping, Still) using accelerometer and gyroscope data.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Loading and Preprocessing

In [ ]:
def load_sensor_data(activity_folder):
    """
    Load accelerometer and gyroscope data from a folder
    
    Args:
        activity_folder: Path to folder containing Accelerometer.csv and Gyroscope.csv
    
    Returns:
        Combined sensor data as DataFrame
    """
    try:
        accel = pd.read_csv(os.path.join(activity_folder, 'Accelerometer.csv'))
        gyro = pd.read_csv(os.path.join(activity_folder, 'Gyroscope.csv'))
        
        # Rename columns to avoid conflicts
        accel_cols = {col: f'accel_{col}' if col not in ['time', 'seconds_elapsed'] else col 
                     for col in accel.columns}
        gyro_cols = {col: f'gyro_{col}' if col not in ['time', 'seconds_elapsed'] else col 
                    for col in gyro.columns}
        
        accel = accel.rename(columns=accel_cols)
        gyro = gyro.rename(columns=gyro_cols)
        
        # Merge on time
        merged = pd.merge(accel, gyro, on='time', suffixes=('_accel', '_gyro'))
        return merged
    except Exception as e:
        print(f"Error loading {activity_folder}: {e}")
        return None


def load_dataset(data_dir, activity_type='Standing'):
    """
    Load all samples for a given activity
    
    Args:
        data_dir: Path to Data directory
        activity_type: Type of activity (Standing, Walking, Jumping, Still)
    
    Returns:
        List of DataFrames
    """
    activity_path = os.path.join(data_dir, activity_type)
    samples = []
    
    if not os.path.exists(activity_path):
        print(f"Activity path not found: {activity_path}")
        return samples
    
    folders = sorted([f for f in os.listdir(activity_path) 
                     if os.path.isdir(os.path.join(activity_path, f))])
    
    for folder in folders:
        data = load_sensor_data(os.path.join(activity_path, folder))
        if data is not None:
            samples.append(data)
    
    return samples


# Load training data
data_dir = r'c:\Users\awini\Formative2_Hidden-Markov-Models\Data'
activities = ['Standing', 'Walking', 'Jumping', 'Still']

training_data = {}
for activity in activities:
    training_data[activity] = load_dataset(data_dir, activity)
    print(f"{activity}: {len(training_data[activity])} samples loaded")

## 3. Feature Extraction (Time & Frequency Domain)

In [ ]:
def extract_time_domain_features(signal):
    """
    Extract time-domain features from signal
    
    Features:
    - Mean
    - Standard Deviation
    - RMS (Root Mean Square)
    - Variance
    - Min, Max, Range
    """
    return {
        'mean': np.mean(signal),
        'std': np.std(signal),
        'rms': np.sqrt(np.mean(signal**2)),
        'var': np.var(signal),
        'min': np.min(signal),
        'max': np.max(signal),
        'range': np.max(signal) - np.min(signal),
        'kurtosis': pd.Series(signal).kurtosis(),
        'skewness': pd.Series(signal).skew()
    }


def extract_frequency_domain_features(signal, sampling_rate=100):
    """
    Extract frequency-domain features using FFT
    
    Features:
    - Dominant Frequency
    - Spectral Energy
    - Max Power
    """
    freqs, power = welch(signal, fs=sampling_rate, nperseg=min(256, len(signal)))
    
    # Avoid division by zero
    if np.sum(power) == 0:
        return {
            'dominant_freq': 0,
            'spectral_energy': 0,
            'max_power': 0,
            'centroid_freq': 0
        }
    
    dominant_idx = np.argmax(power)
    spectral_energy = np.sum(power)
    
    # Spectral centroid
    centroid = np.sum(freqs * power) / spectral_energy if spectral_energy > 0 else 0
    
    return {
        'dominant_freq': freqs[dominant_idx],
        'spectral_energy': spectral_energy,
        'max_power': power[dominant_idx],
        'centroid_freq': centroid
    }


def extract_correlation_features(accel_x, accel_y, accel_z, gyro_x, gyro_y, gyro_z):
    """
    Extract correlation features between axes
    """
    accel_data = np.array([accel_x, accel_y, accel_z])
    gyro_data = np.array([gyro_x, gyro_y, gyro_z])
    
    # Correlation between accelerometer axes
    accel_corr = np.corrcoef(accel_data)
    gyro_corr = np.corrcoef(gyro_data)
    
    return {
        'accel_xy_corr': accel_corr[0, 1],
        'accel_xz_corr': accel_corr[0, 2],
        'accel_yz_corr': accel_corr[1, 2],
        'gyro_xy_corr': gyro_corr[0, 1],
        'gyro_xz_corr': gyro_corr[0, 2],
        'gyro_yz_corr': gyro_corr[1, 2]
    }


def extract_window_features(data_window, window_name="", sampling_rate=100):
    """
    Extract all features from a data window
    
    Args:
        data_window: DataFrame with sensor data
        window_name: Name of window (for debugging)
        sampling_rate: Sampling rate in Hz
    
    Returns:
        Dictionary of features
    """
    features = {}
    
    try:
        # Get sensor columns
        accel_cols = [col for col in data_window.columns if col.startswith('accel_') and col[-1] in 'xyz']
        gyro_cols = [col for col in data_window.columns if col.startswith('gyro_') and col[-1] in 'xyz']
        
        # Time-domain features for each axis
        for col in accel_cols:
            axis_name = col.split('_')[-1]
            features[f'accel_{axis_name}_mean'] = np.mean(data_window[col])
            features[f'accel_{axis_name}_std'] = np.std(data_window[col])
            features[f'accel_{axis_name}_rms'] = np.sqrt(np.mean(data_window[col]**2))
        
        for col in gyro_cols:
            axis_name = col.split('_')[-1]
            features[f'gyro_{axis_name}_mean'] = np.mean(data_window[col])
            features[f'gyro_{axis_name}_std'] = np.std(data_window[col])
            features[f'gyro_{axis_name}_rms'] = np.sqrt(np.mean(data_window[col]**2))
        
        # Frequency-domain features for magnitude
        accel_mag = np.sqrt(data_window['accel_x']**2 + data_window['accel_y']**2 + data_window['accel_z']**2)
        gyro_mag = np.sqrt(data_window['gyro_x']**2 + data_window['gyro_y']**2 + data_window['gyro_z']**2)
        
        freq_accel = extract_frequency_domain_features(accel_mag, sampling_rate)
        freq_gyro = extract_frequency_domain_features(gyro_mag, sampling_rate)
        
        for key, val in freq_accel.items():
            features[f'accel_mag_{key}'] = val
        for key, val in freq_gyro.items():
            features[f'gyro_mag_{key}'] = val
        
        # Correlation features
        corr_feat = extract_correlation_features(
            data_window['accel_x'], data_window['accel_y'], data_window['accel_z'],
            data_window['gyro_x'], data_window['gyro_y'], data_window['gyro_z']
        )
        features.update(corr_feat)
        
        # Signal Magnitude Area (SMA)
        sma_accel = np.mean(np.abs(accel_mag))
        sma_gyro = np.mean(np.abs(gyro_mag))
        features['accel_sma'] = sma_accel
        features['gyro_sma'] = sma_gyro
        
        return features
    except Exception as e:
        print(f"Error extracting features from {window_name}: {e}")
        return {}


def create_feature_matrix(sensor_data, window_size=100, sampling_rate=100):
    """
    Create feature matrix by sliding window over sensor data
    
    Args:
        sensor_data: DataFrame with sensor readings
        window_size: Size of sliding window (samples)
        sampling_rate: Sampling rate in Hz
    
    Returns:
        DataFrame with features for each window
    """
    features_list = []
    
    if len(sensor_data) < window_size:
        # If data is shorter than window, use all data
        feat = extract_window_features(sensor_data, sampling_rate=sampling_rate)
        if feat:
            features_list.append(feat)
    else:
        # Use sliding window with 50% overlap
        step = window_size // 2
        for i in range(0, len(sensor_data) - window_size + 1, step):
            window = sensor_data.iloc[i:i+window_size]
            feat = extract_window_features(window, sampling_rate=sampling_rate)
            if feat:
                features_list.append(feat)
    
    return pd.DataFrame(features_list)


print("Feature extraction functions defined.")

## 4. Prepare Training Data

In [ ]:
# Extract features for all training samples
train_features_by_activity = {}
train_sequences_by_activity = {}

for activity in activities:
    all_features = []
    all_sequences = []  # Store time series for each sample
    
    for sample_data in training_data[activity]:
        # Create feature matrix from this sample
        features = create_feature_matrix(sample_data, window_size=100)
        if len(features) > 0:
            all_features.append(features)
            all_sequences.append(features.values)  # Store raw feature values
    
    if all_features:
        train_features_by_activity[activity] = pd.concat(all_features, ignore_index=True)
        train_sequences_by_activity[activity] = all_sequences
        print(f"{activity}: {len(all_features)} samples, {len(train_features_by_activity[activity])} feature windows")
    else:
        print(f"{activity}: NO FEATURES EXTRACTED")

## 5. Normalize Features

In [ ]:
# Fit scaler on all training data
all_train_features = pd.concat(train_features_by_activity.values(), ignore_index=True)
scaler = StandardScaler()
scaler.fit(all_train_features)

print(f"Scaler fitted on {len(all_train_features)} feature vectors")
print(f"Number of features: {len(all_train_features.columns)}")
print(f"\nFeature columns: {all_train_features.columns.tolist()[:5]}...")  # Show first 5

# Normalize training sequences
normalized_sequences = {}
for activity in activities:
    if activity in train_sequences_by_activity:
        normalized = []
        for seq in train_sequences_by_activity[activity]:
            norm_seq = scaler.transform(seq)
            normalized.append(norm_seq)
        normalized_sequences[activity] = normalized

## 6. HMM Implementation

In [ ]:
class GaussianHMM:
    """
    Gaussian Hidden Markov Model implementation
    """
    
    def __init__(self, n_states=3, n_features=None, random_state=42):
        self.n_states = n_states
        self.n_features = n_features
        self.random_state = random_state
        np.random.seed(random_state)
        
        # Initialize parameters
        self.transition_matrix = None
        self.means = None
        self.covariances = None
        self.initial_probs = None
        self.trained = False
    
    def initialize_parameters(self, sequences):
        """
        Initialize HMM parameters
        """
        # Concatenate all sequences
        all_data = np.vstack(sequences)
        
        # Random initialization of state means
        indices = np.random.choice(len(all_data), self.n_states, replace=False)
        self.means = all_data[indices].copy()
        
        # Initialize covariances as identity times variance
        total_var = np.var(all_data, axis=0)
        self.covariances = np.array([np.diag(total_var) for _ in range(self.n_states)])
        
        # Uniform transition matrix
        self.transition_matrix = np.ones((self.n_states, self.n_states)) / self.n_states
        
        # Uniform initial probabilities
        self.initial_probs = np.ones(self.n_states) / self.n_states
    
    def gaussian_pdf(self, x, mean, cov):
        """
        Compute Gaussian probability density
        """
        n = len(x)
        det = np.linalg.det(cov)
        if det <= 0:
            det = 1e-10
        
        diff = x - mean
        inv_cov = np.linalg.inv(cov + np.eye(n) * 1e-6)
        exponent = -0.5 * diff.T @ inv_cov @ diff
        coeff = 1.0 / np.sqrt((2 * np.pi) ** n * abs(det))
        return coeff * np.exp(exponent)
    
    def forward_pass(self, sequence):
        """
        Forward algorithm - compute forward probabilities
        """
        T = len(sequence)
        alphas = np.zeros((T, self.n_states))
        
        # Initialize
        for j in range(self.n_states):
            alphas[0, j] = self.initial_probs[j] * self.gaussian_pdf(
                sequence[0], self.means[j], self.covariances[j]
            )
        
        # Recursion
        for t in range(1, T):
            for j in range(self.n_states):
                alphas[t, j] = self.gaussian_pdf(sequence[t], self.means[j], self.covariances[j]) * \
                               np.sum(alphas[t-1, :] * self.transition_matrix[:, j])
        
        return alphas
    
    def backward_pass(self, sequence):
        """
        Backward algorithm - compute backward probabilities
        """
        T = len(sequence)
        betas = np.zeros((T, self.n_states))
        
        # Initialize
        betas[T-1, :] = 1
        
        # Recursion backwards
        for t in range(T-2, -1, -1):
            for i in range(self.n_states):
                betas[t, i] = np.sum(
                    self.transition_matrix[i, :] * self.gaussian_pdf(
                        sequence[t+1], self.means[:], self.covariances[:]
                    ).reshape(-1) * betas[t+1, :]
                )
        
        return betas
    
    def viterbi(self, sequence):
        """
        Viterbi algorithm - find most likely state sequence
        """
        T = len(sequence)
        viterbi_probs = np.zeros((T, self.n_states))
        backpointers = np.zeros((T, self.n_states), dtype=int)
        
        # Initialize
        for j in range(self.n_states):
            viterbi_probs[0, j] = self.initial_probs[j] * self.gaussian_pdf(
                sequence[0], self.means[j], self.covariances[j]
            )
        
        # Recursion
        for t in range(1, T):
            for j in range(self.n_states):
                candidates = viterbi_probs[t-1, :] * self.transition_matrix[:, j]
                backpointers[t, j] = np.argmax(candidates)
                viterbi_probs[t, j] = np.max(candidates) * self.gaussian_pdf(
                    sequence[t], self.means[j], self.covariances[j]
                )
        
        # Backtrack
        path = np.zeros(T, dtype=int)
        path[T-1] = np.argmax(viterbi_probs[T-1, :])
        for t in range(T-2, -1, -1):
            path[t] = backpointers[t+1, path[t+1]]
        
        return path, np.max(viterbi_probs[T-1, :])
    
    def baum_welch(self, sequences, n_iterations=5, tolerance=1e-3):
        """
        Baum-Welch algorithm - train HMM parameters
        """
        self.initialize_parameters(sequences)
        
        prev_log_likelihood = -np.inf
        
        for iteration in range(n_iterations):
            # E-step: compute forward and backward probabilities
            numerator_transition = np.zeros((self.n_states, self.n_states))
            denominator_transition = np.zeros(self.n_states)
            
            numerator_means = np.zeros((self.n_states, self.n_features))
            numerator_covar = np.zeros((self.n_states, self.n_features, self.n_features))
            denominator_gamma = np.zeros(self.n_states)
            
            total_log_likelihood = 0
            
            for sequence in sequences:
                alphas = self.forward_pass(sequence)
                betas = self.backward_pass(sequence)
                
                # Normalize
                scale = np.sum(alphas[-1, :])
                if scale <= 0:
                    scale = 1e-10
                total_log_likelihood += np.log(scale)
                
                # Compute gammas and xis
                T = len(sequence)
                
                for t in range(T-1):
                    denom = np.sum(alphas[t, :] * betas[t, :])
                    if denom <= 0:
                        denom = 1e-10
                    
                    for i in range(self.n_states):
                        gamma = (alphas[t, i] * betas[t, i]) / denom
                        denominator_gamma[i] += gamma
                        numerator_means[i] += gamma * sequence[t]
                        
                        diff = sequence[t] - self.means[i]
                        numerator_covar[i] += gamma * np.outer(diff, diff)
                        
                        for j in range(self.n_states):
                            xi = (alphas[t, i] * self.transition_matrix[i, j] * 
                                  self.gaussian_pdf(sequence[t+1], self.means[j], self.covariances[j]) * 
                                  betas[t+1, j]) / denom
                            numerator_transition[i, j] += xi
                            denominator_transition[i] += xi
            
            # M-step: update parameters
            for i in range(self.n_states):
                if denominator_gamma[i] > 0:
                    self.means[i] = numerator_means[i] / denominator_gamma[i]
                    self.covariances[i] = numerator_covar[i] / denominator_gamma[i] + np.eye(self.n_features) * 1e-6
                
                if denominator_transition[i] > 0:
                    self.transition_matrix[i, :] = numerator_transition[i, :] / denominator_transition[i]
            
            # Check convergence
            log_likelihood_change = (total_log_likelihood - prev_log_likelihood) / len(sequences)
            if iteration % 1 == 0:
                print(f"Iteration {iteration+1}: Log-likelihood = {total_log_likelihood:.4f}")
            
            if abs(log_likelihood_change) < tolerance:
                print(f"Converged at iteration {iteration+1}")
                break
            
            prev_log_likelihood = total_log_likelihood
        
        self.trained = True
    
    def predict(self, sequence):
        """
        Predict state sequence using Viterbi
        """
        if not self.trained:
            raise ValueError("Model must be trained first")
        return self.viterbi(sequence)[0]


print("GaussianHMM class defined.")

## 7. Train HMM Models

In [ ]:
# Train one model per activity
models = {}
n_features = train_features_by_activity[activities[0]].shape[1]

for activity in activities:
    print(f"\n=== Training HMM for {activity} ===")
    model = GaussianHMM(n_states=3, n_features=n_features)
    
    if activity in normalized_sequences:
        model.baum_welch(normalized_sequences[activity], n_iterations=10)
        models[activity] = model
    else:
        print(f"No sequences found for {activity}")

print("\nAll models trained!")

## 8. Visualization: Transition Matrices

In [ ]:
# Visualize transition matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, activity in enumerate(activities):
    if activity in models:
        trans_matrix = models[activity].transition_matrix
        sns.heatmap(trans_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
                   ax=axes[idx], cbar=True, square=True,
                   xticklabels=[f'S{i}' for i in range(trans_matrix.shape[0])],
                   yticklabels=[f'S{i}' for i in range(trans_matrix.shape[0])])
        axes[idx].set_title(f'{activity} - Transition Matrix')
        axes[idx].set_xlabel('To State')
        axes[idx].set_ylabel('From State')

plt.tight_layout()
plt.savefig('transition_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTransition matrices saved as 'transition_matrices.png'")

## 9. Load and Evaluate on Test Data

In [ ]:
# Load test data
test_data = {}
for activity in activities:
    test_data[activity] = load_dataset(data_dir, f"{activity}_test")
    print(f"{activity} test: {len(test_data[activity])} samples")


# Extract features for test data
test_features_by_activity = {}
test_sequences_by_activity = {}
test_labels_by_activity = {}

for activity in activities:
    all_labels = []
    all_sequences = []
    activity_idx = activities.index(activity)
    
    for sample_data in test_data[activity]:
        features = create_feature_matrix(sample_data, window_size=100)
        if len(features) > 0:
            # Normalize
            normalized_feat = scaler.transform(features)
            all_sequences.append(normalized_feat)
            # Label all windows as this activity
            all_labels.extend([activity_idx] * len(normalized_feat))
    
    if all_sequences:
        test_sequences_by_activity[activity] = all_sequences
        test_labels_by_activity[activity] = all_labels
        print(f"{activity} test features extracted: {len(all_labels)} windows")

print("\nTest data prepared!")

## 10. Model Evaluation

In [ ]:
# Evaluate on test data
all_true_labels = []
all_predictions = []

for activity_idx, activity in enumerate(activities):
    if activity in test_sequences_by_activity:
        print(f"\nEvaluating {activity}...")
        
        for test_seq in test_sequences_by_activity[activity]:
            # Predict state sequence
            predicted_states = models[activity].predict(test_seq)
            
            # For classification: use most common state in the prediction
            unique, counts = np.unique(predicted_states, return_counts=True)
            predicted_activity_state = unique[np.argmax(counts)]
            
            # Actually, we need a different approach - use all models to score
            # and pick the activity with best likelihood
            break  # We'll fix this in the next cell

print("\nInitializing multi-model evaluation...")

## 11. Multi-Model Scoring & Classification

In [ ]:
def calculate_sequence_likelihood(model, sequence):
    """
    Calculate likelihood of a sequence given the model
    """
    alphas = model.forward_pass(sequence)
    likelihood = np.sum(alphas[-1, :])
    return likelihood


def classify_sequence(sequence, models, scaler):
    """
    Classify a sequence by finding which activity model gives highest likelihood
    """
    best_activity = None
    best_likelihood = -np.inf
    likelihoods = {}
    
    for activity, model in models.items():
        likelihood = calculate_sequence_likelihood(model, sequence)
        likelihoods[activity] = likelihood
        
        if likelihood > best_likelihood:
            best_likelihood = likelihood
            best_activity = activity
    
    return best_activity, likelihoods


# Evaluate
confusion_data = {activity: [] for activity in activities}
true_activity_labels = []
predicted_activity_labels = []

for true_activity in activities:
    if true_activity not in test_sequences_by_activity:
        continue
    
    print(f"\nProcessing {true_activity} test samples...")
    true_activity_idx = activities.index(true_activity)
    
    for seq_idx, test_seq in enumerate(test_sequences_by_activity[true_activity]):
        pred_activity, likelihoods = classify_sequence(test_seq, models, scaler)
        
        true_activity_labels.append(true_activity)
        predicted_activity_labels.append(pred_activity)
        confusion_data[true_activity].append(pred_activity)
        
        if seq_idx < 3:  # Print first 3
            print(f"  Sample {seq_idx+1} (True: {true_activity})")
            for act, lik in sorted(likelihoods.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"    {act}: {lik:.4f}")

print("\nEvaluation complete!")

## 12. Performance Metrics

In [ ]:
# Convert to indices for confusion matrix
true_indices = [activities.index(a) for a in true_activity_labels]
pred_indices = [activities.index(a) for a in predicted_activity_labels]

# Overall accuracy
overall_accuracy = accuracy_score(true_indices, pred_indices)
print(f"Overall Accuracy: {overall_accuracy:.4f}")

# Per-activity metrics
metrics_data = []
for idx, activity in enumerate(activities):
    # True positives, false positives, false negatives, true negatives
    tp = sum((np.array(true_indices) == idx) & (np.array(pred_indices) == idx))
    fp = sum((np.array(true_indices) != idx) & (np.array(pred_indices) == idx))
    fn = sum((np.array(true_indices) == idx) & (np.array(pred_indices) != idx))
    tn = sum((np.array(true_indices) != idx) & (np.array(pred_indices) != idx))
    
    # Sensitivity (recall/TPR)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Specificity (TNR)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Precision
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    n_samples = sum(np.array(true_indices) == idx)
    accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    
    metrics_data.append({
        'Activity': activity,
        'Samples': n_samples,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Precision': precision,
        'Accuracy': accuracy
    })

metrics_df = pd.DataFrame(metrics_data)
print("\n=== Performance Metrics by Activity ===")
print(metrics_df.to_string(index=False))

# Save metrics
metrics_df.to_csv('performance_metrics.csv', index=False)
print("\nMetrics saved to 'performance_metrics.csv'")

## 13. Confusion Matrix Visualization

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(true_indices, pred_indices, labels=range(len(activities)))

# Normalize for better visualization
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
           xticklabels=activities, yticklabels=activities, cbar=True)
ax1.set_title('Confusion Matrix (Raw Counts)')
ax1.set_ylabel('True Activity')
ax1.set_xlabel('Predicted Activity')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
           xticklabels=activities, yticklabels=activities, cbar=True, vmin=0, vmax=1)
ax2.set_title('Confusion Matrix (Normalized)')
ax2.set_ylabel('True Activity')
ax2.set_xlabel('Predicted Activity')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nConfusion matrix saved as 'confusion_matrix.png'")

## 14. Visualization: Sample Predictions

In [ ]:
# Show a few predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for activity_idx, activity in enumerate(activities):
    if activity in test_sequences_by_activity and len(test_sequences_by_activity[activity]) > 0:
        test_seq = test_sequences_by_activity[activity][0][:50]  # First 50 samples
        
        # Get predictions from true activity model
        true_model = models[activity]
        pred_states = true_model.predict(test_seq)
        
        # Also get predictions from other models
        other_predictions = {}
        for other_activity in activities:
            if other_activity != activity:
                other_model = models[other_activity]
                other_pred = other_model.predict(test_seq)
                other_predictions[other_activity] = other_pred
        
        # Plot
        axes[activity_idx].plot(pred_states, 'o-', label=f'{activity} model', linewidth=2, markersize=4)
        axes[activity_idx].set_title(f'Predicted states for {activity}')
        axes[activity_idx].set_xlabel('Time Window')
        axes[activity_idx].set_ylabel('Hidden State')
        axes[activity_idx].set_ylim(-0.5, 2.5)
        axes[activity_idx].grid(alpha=0.3)
        axes[activity_idx].legend()

plt.tight_layout()
plt.savefig('state_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nState predictions visualization saved as 'state_predictions.png'")

## 15. Sample Feature Visualization

In [ ]:
# Visualize sample raw sensor data
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, activity in enumerate(activities):
    if activity in training_data and len(training_data[activity]) > 0:
        sample_data = training_data[activity][0]
        
        # Plot accelerometer
        time = np.arange(len(sample_data))
        axes[idx].plot(time, sample_data['accel_x'], label='Accel X', alpha=0.7)
        axes[idx].plot(time, sample_data['accel_y'], label='Accel Y', alpha=0.7)
        axes[idx].plot(time, sample_data['accel_z'], label='Accel Z', alpha=0.7)
        
        axes[idx].set_title(f'{activity} - Sample Raw Accelerometer Data')
        axes[idx].set_xlabel('Sample')
        axes[idx].set_ylabel('Acceleration (g)')
        axes[idx].legend()
        axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('raw_sensor_data.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nRaw sensor data visualization saved as 'raw_sensor_data.png'")

## 16. Summary

In [ ]:
print("="*60)
print("HIDDEN MARKOV MODEL - ACTIVITY RECOGNITION SUMMARY")
print("="*60)

print("\n1. DATA COLLECTION")
print(f"   - Total activities: {len(activities)}")
for activity in activities:
    n_train = len(training_data.get(activity, []))
    n_test = len(test_data.get(activity, []))
    print(f"   - {activity}: {n_train} training, {n_test} test samples")

print("\n2. FEATURE EXTRACTION")
print(f"   - Total features extracted: {n_features}")
print("   - Time-domain: mean, std, RMS for each axis")
print("   - Frequency-domain: dominant frequency, spectral energy")
print("   - Others: correlation, SMA")

print("\n3. HMM CONFIGURATION")
print(f"   - Hidden states per model: 3")
print(f"   - Number of models: {len(models)}")
print(f"   - Algorithm: Baum-Welch (EM)")

print("\n4. RESULTS")
print(f"   - Overall accuracy: {overall_accuracy:.2%}")
print(f"   - Confusion matrix generated")
print(f"   - Per-activity metrics computed")

print("\n5. OUTPUTS GENERATED")
print("   - transition_matrices.png")
print("   - confusion_matrix.png")
print("   - state_predictions.png")
print("   - raw_sensor_data.png")
print("   - performance_metrics.csv")

print("\n" + "="*60)